In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor


train_path = r"C:\Users\LENOVO\Downloads\train.csv"
test_path  = r"C:\Users\LENOVO\Downloads\test.csv"
sub_path   = r"C:\Users\LENOVO\Downloads\sample_submission_file.csv"

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

for df in (train, test):
    df["service_date"] = pd.to_datetime(df["service_date"], dayfirst=True)
    df["dow"] = df["service_date"].dt.weekday          # 0-6
    df["month"] = df["service_date"].dt.month          # 1-12
    df["day"] = df["service_date"].dt.day              # 1-31
    df["weekofyear"] = df["service_date"].dt.isocalendar().week.astype(int)
    df["is_weekend"] = (df["dow"] >= 5).astype(int)

origin_mean = train.groupby("origin_hub_id")["final_service_units"].mean()
train["origin_avg"] = train["origin_hub_id"].map(origin_mean)
test["origin_avg"]  = test["origin_hub_id"].map(origin_mean).fillna(origin_mean.mean())

dest_mean = train.groupby("destination_hub_id")["final_service_units"].mean()
train["dest_avg"] = train["destination_hub_id"].map(dest_mean)
test["dest_avg"]  = test["destination_hub_id"].map(dest_mean).fillna(dest_mean.mean())

train["route_id"] = train["origin_hub_id"].astype(str) + "_" + train["destination_hub_id"].astype(str)
test["route_id"]  = test["origin_hub_id"].astype(str) + "_" + test["destination_hub_id"].astype(str)

route_mean = train.groupby("route_id")["final_service_units"].mean()
global_mean = train["final_service_units"].mean()

train["route_avg"] = train["route_id"].map(route_mean)
test["route_avg"]  = test["route_id"].map(route_mean).fillna(global_mean)

route_dow_mean = train.groupby(["route_id","dow"])["final_service_units"].mean()
train["route_dow_avg"] = train.set_index(["route_id","dow"]).index.map(route_dow_mean)
test["route_dow_avg"]  = test.set_index(["route_id","dow"]).index.map(route_dow_mean)

train["route_dow_avg"] = train["route_dow_avg"].fillna(train["route_avg"].fillna(global_mean))
test["route_dow_avg"]  = test["route_dow_avg"].fillna(test["route_avg"].fillna(global_mean))

FEATURES = [
    "origin_hub_id","destination_hub_id",
    "dow","month","day","weekofyear","is_weekend",
    "origin_avg","dest_avg","route_avg","route_dow_avg"
]

X = train[FEATURES]
y = train["final_service_units"]
X_test = test[FEATURES]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds_lgb = np.zeros(len(train))
oof_preds_xgb = np.zeros(len(train))
oof_preds_cat = np.zeros(len(train))

test_preds_lgb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))
test_preds_cat = np.zeros(len(test))

fold_mae_lgb = []
fold_mae_xgb = []
fold_mae_cat = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y), 1):
    print(f"\n===== FOLD {fold} =====")
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    lgb = LGBMRegressor(
        n_estimators=2500,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_samples=40,
        reg_lambda=1.0,
        random_state=42
    )
    lgb.fit(X_tr, y_tr)
    val_pred_lgb = lgb.predict(X_val)
    oof_preds_lgb[val_idx] = val_pred_lgb
    mae_lgb = mean_absolute_error(y_val, val_pred_lgb)
    fold_mae_lgb.append(mae_lgb)
    print("LGBM MAE:", mae_lgb)

    test_preds_lgb += lgb.predict(X_test) / kf.n_splits

    xgb = XGBRegressor(
        n_estimators=2500,
        learning_rate=0.03,
        max_depth=7,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        objective="reg:squarederror",
        n_jobs=-1,
        random_state=42
    )
    xgb.fit(X_tr, y_tr)
    val_pred_xgb = xgb.predict(X_val)
    oof_preds_xgb[val_idx] = val_pred_xgb
    mae_xgb = mean_absolute_error(y_val, val_pred_xgb)
    fold_mae_xgb.append(mae_xgb)
    print("XGB MAE:", mae_xgb)

    test_preds_xgb += xgb.predict(X_test) / kf.n_splits

    cat = CatBoostRegressor(
        iterations=2500,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=5.0,
        loss_function="MAE",
        random_seed=42,
        verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
    val_pred_cat = cat.predict(X_val)
    oof_preds_cat[val_idx] = val_pred_cat
    mae_cat = mean_absolute_error(y_val, val_pred_cat)
    fold_mae_cat.append(mae_cat)
    print("CAT MAE:", mae_cat)

    test_preds_cat += cat.predict(X_test) / kf.n_splits

print("OOF METRICS")
print("LGBM OOF MAE:", mean_absolute_error(y, oof_preds_lgb))
print("XGB  OOF MAE:", mean_absolute_error(y, oof_preds_xgb))
print("CAT  OOF MAE:", mean_absolute_error(y, oof_preds_cat))

oof_ens = 0.4 * oof_preds_lgb + 0.3 * oof_preds_xgb + 0.3 * oof_preds_cat
print("ENSEMBLE OOF MAE:", mean_absolute_error(y, oof_ens))

r2_lgb = r2_score(y, oof_preds_lgb)
r2_xgb = r2_score(y, oof_preds_xgb)
r2_cat = r2_score(y, oof_preds_cat)
r2_ens = r2_score(y, oof_ens)

print("OOF R^2")
print("LGBM OOF R^2:", r2_lgb)
print("XGB  OOF R^2:", r2_xgb)
print("CAT  OOF R^2:", r2_cat)
print("ENSEMBLE OOF R^2:", r2_ens)

tolerance = 5.0  

acc_lgb = np.mean(np.abs(y - oof_preds_lgb) <= tolerance)
acc_xgb = np.mean(np.abs(y - oof_preds_xgb) <= tolerance)
acc_cat = np.mean(np.abs(y - oof_preds_cat) <= tolerance)
acc_ens = np.mean(np.abs(y - oof_ens) <= tolerance)

print(f"\n OOF 'ACCURACY' (|y - y_pred| <= {tolerance})")
print("LGBM OOF accuracy:", acc_lgb)
print("XGB  OOF accuracy:", acc_xgb)
print("CAT  OOF accuracy:", acc_cat)
print("ENSEMBLE OOF accuracy:", acc_ens)


test_pred_final = 0.4 * test_preds_lgb + 0.3 * test_preds_xgb + 0.3 * test_preds_cat

sub = pd.read_csv(sub_path)
sub["final_service_units"] = test_pred_final

out_path = "my_submission_day2_lgb_xgb_cat.csv"
sub.to_csv(out_path, index=False)
print("Saved submission to:", out_path)



===== FOLD 1 =====
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002631 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 647
[LightGBM] [Info] Number of data points in the train set: 53760, number of used features: 11
[LightGBM] [Info] Start training from score 2003.632533
LGBM MAE: 302.2446890641174
XGB MAE: 303.7242736816406
CAT MAE: 337.3925057329719

===== FOLD 2 =====
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001606 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 646
[LightGBM] [Info] Number of data points in the train set: 53760, number of used features: 11
[LightGBM] [Info] Start training from score 1998.978813
LGBM MAE: 308.85574243088615
XGB MAE: 312.5091247558594
CAT MA